In [1]:
import numpy as np

from scipy.special import comb as n_choose_k
from sklearn.linear_model import LogisticRegression
from scipy.stats import pearsonr, spearmanr
from warnings import simplefilter
from sklearn.exceptions import ConvergenceWarning
simplefilter("ignore", category=ConvergenceWarning)


import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from tqdm import tqdm

from itertools import combinations
from collections import defaultdict
import random

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda')
device

from src.data import *

from sklearn.decomposition import DictionaryLearning
from sklearn.decomposition import sparse_encode

In [2]:
# Parameters
seed = 7012025
C = np.inf # no regularisation for lin. regression
power = 1 # uniform density
D = 1000 # number of data samples
N = 100 # number of sources
K = 10 # sparsity
num_ood = N // 2 # how many new OOD sources
M = int(np.ceil(K * np.log(N / K) * 1)) # Compressed Sensing bound
num_ood, M

(50, 24)

In [3]:
# lazy example (might not be perfect), just draw random A
np.random.seed(seed)
# https://en.wikipedia.org/wiki/Restricted_isometry_property
A = np.random.normal(0, 1, (M, N)) # random normal has RIP -> CS works =)
A /= np.linalg.norm(A, axis=0, keepdims=True)
A.min(), A.max(), A.shape

(np.float64(-0.6458119825139067), np.float64(0.6527911931680838), (24, 100))

In [4]:
# generate data
np.random.seed(seed)

Z_iid = np.array([sample_iid(n=N, k=K) for _ in range(D)])
Y_iid = Z_iid @ A.T
label_iid = Z_iid[:, 0] > .5

Z_ood = np.array([sample_ood(n=N, k=K) for _ in range(D)])
Y_ood = Z_ood @ A.T
label_ood = Z_ood[:, 0] > .5

Z_iid.shape, Y_iid.shape, label_iid[:10], np.mean(label_iid)

((1000, 100),
 (1000, 24),
 array([False, False, False, False, False, False,  True, False, False,
        False]),
 np.float64(0.237))

In [5]:
Y_all = np.vstack([Y_iid, Y_ood])
Z_all = np.vstack([Z_iid, Z_ood])

In [6]:
Y_all.shape

(2000, 24)

In [7]:
Z_all.shape

(2000, 100)

# supervised

In [8]:
num_seed = 1

# target = np.concatenate([z_01, z_12, z_02], 0)
# inputs = target @ A
# inputs = torch.tensor(inputs, dtype=torch.float32, device=device)
# target = torch.tensor(target, dtype=torch.float32, device=device)

inputs = torch.tensor(Y_all, dtype=torch.float32, device=device)

best_loss = np.inf
best_D, best_Z = None, None

d_x = inputs.shape[1]
d_z = Z_iid.shape[1]
n_points = inputs.shape[0]

for rep in range(num_seed):
    torch.manual_seed(seed + rep)

    log_Z = torch.randn(n_points, d_z, device=device).requires_grad_()
    D = torch.randn(d_z, d_x, device=device).requires_grad_()
    # D = torch.tensor(A, dtype=torch.float32, device=device)
    optim = torch.optim.Adam([log_Z, D], lr=1e-2)

    for i in tqdm(range(100000)):
        # Z = torch.exp(log_Z)
        Z = torch.nn.functional.softplus(log_Z)
        # Z = torch.nn.functional.relu(log_Z)
        # Z = torch.nn.functional.gelu(log_Z)
        rec = Z @ D
        # print(rec.shape)
        mse = torch.mean((inputs - rec)**2)
        # l1 = torch.mean(Z)
        l1 = torch.mean(torch.abs(Z) * torch.linalg.norm(D, dim=1))
        loss = mse + 0.001 * l1
        optim.zero_grad()
        loss.backward()
        optim.step()

        # D.data /= torch.linalg.norm(D, dim=1, keepdim=True)
        
        # if rep == 0 and not i % 10000:
        #     print(rep, i, mse.item(), l1.item())
        #     fig = plt.figure()
        #     ax = fig.add_subplot(projection='3d')
        #     ax.scatter(*Z.detach().cpu().numpy().T,)
        #     ax.view_init(elev=30, azim=45, roll=0)
        #     plt.show()

        if i > 10000:
            if loss.item() > best_loss * 1.1:
                break
    print(rep, i, mse.item(), l1.item())
    # fig = plt.figure()
    # ax = fig.add_subplot(projection='3d')
    # ax.scatter(*Z.detach().cpu().numpy().T, s=5)
    # ax.view_init(elev=30, azim=45, roll=0)
    # plt.show()
    if loss.item() < best_loss:
        best_D = D.detach().cpu().numpy()
        best_Z = Z.detach().cpu().numpy()

100%|██████████| 100000/100000 [02:34<00:00, 645.48it/s]

0 99999 1.6299827620969154e-06 0.055065739899873734


In [9]:
device

device(type='cuda')

In [10]:
from src.mcc import mean_corr_coef as mcc

/grid/klindt/home/barinpac/.conda/envs/jhub-py312/lib/python3.12/site-packages/torch/__init__.py:1144: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647378361/work/torch/csrc/tensor/python_tensor.cpp:432.)
  _C._set_default_tensor_type(t)


In [11]:
mcc(best_Z, Z_all)

np.float64(0.27854462301967287)

In [15]:
from src.evaluation import compute_mcc

In [16]:
from src.mcc import get_mcc_np

In [17]:
compute_mcc(get_mcc_np(best_Z, Z_all))

np.float64(0.27854462301967287)

# unsupervised

In [21]:
num_seed = 10

# target = np.concatenate([z_01, z_12, z_02], 0)
# inputs = target @ A
# inputs = torch.tensor(inputs, dtype=torch.float32, device=device)
# target = torch.tensor(target, dtype=torch.float32, device=device)

inputs = torch.tensor(Y_all, dtype=torch.float32, device=device)

best_loss = np.inf
best_D, best_Z = None, None

d_x = inputs.shape[1]
d_z = Z_iid.shape[1]
n_points = inputs.shape[0]

for rep in range(num_seed):
    torch.manual_seed(seed + rep)

    log_Z = torch.randn(n_points, d_z, device=device).requires_grad_()
    D = torch.randn(d_z, d_x, device=device).requires_grad_()
    # D = torch.tensor(A, dtype=torch.float32, device=device)
    optim = torch.optim.Adam([log_Z, D], lr=1e-2)

    for i in tqdm(range(100000)):
        # Z = torch.exp(log_Z)
        Z = torch.nn.functional.softplus(log_Z)
        # Z = torch.nn.functional.relu(log_Z)
        # Z = torch.nn.functional.gelu(log_Z)
        rec = Z @ D
        # print(rec.shape)
        mse = torch.mean((inputs - rec)**2)
        # l1 = torch.mean(Z)
        l1 = torch.mean(torch.abs(Z) * torch.linalg.norm(D, dim=1))
        loss = mse + 0.001 * l1
        optim.zero_grad()
        loss.backward()
        optim.step()

        # D.data /= torch.linalg.norm(D, dim=1, keepdim=True)
        
        # if rep == 0 and not i % 10000:
        #     print(rep, i, mse.item(), l1.item())
        #     fig = plt.figure()
        #     ax = fig.add_subplot(projection='3d')
        #     ax.scatter(*Z.detach().cpu().numpy().T,)
        #     ax.view_init(elev=30, azim=45, roll=0)
        #     plt.show()

        if i > 10000:
            if loss.item() > best_loss * 1.1:
                break
    print(rep, i, mse.item(), l1.item())
    # fig = plt.figure()
    # ax = fig.add_subplot(projection='3d')
    # ax.scatter(*Z.detach().cpu().numpy().T, s=5)
    # ax.view_init(elev=30, azim=45, roll=0)
    # plt.show()
    if loss.item() < best_loss:
        best_D = D.detach().cpu().numpy()
        best_Z = Z.detach().cpu().numpy()

  0%|          | 0/100000 [00:00<?, ?it/s]

100%|██████████| 100000/100000 [01:01<00:00, 1636.89it/s]


0 99999 1.6391356148943892e-06 0.055138823284987426


100%|██████████| 100000/100000 [01:01<00:00, 1615.59it/s]


1 99999 1.5845863834550882e-06 0.05480135046833259


...

7 99999 3.0188592132693626e-06 0.05492506869307046


100%|██████████| 100000/100000 [01:01<00:00, 1627.84it/s]


8 99999 1.5843633646923816e-06 0.05465516153817623


100%|██████████| 100000/100000 [01:00<00:00, 1648.56it/s]

9 99999 5.5897908440529265e-06 0.05545223160001068


In [23]:
mcc(best_Z, Z_all)

np.float64(0.27324421656938286)

In [29]:
mcc(D.T.detach().cpu().numpy(), A, method='cos')

np.float64(0.5669482095336142)

In [31]:
mcc(D.detach().cpu().numpy(), A.T, method='cos')

np.float64(0.2156468335586007)